In [1]:
# 03_band_migration.ipynb
# Level 3 (Theorem 3): band migration rate is linear in the band count K and
# in the perturbation scale. Equal-frequency banding of the continuous score;
# migration = share of applicants whose grade changes after quantization.

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "axes.grid": True,
                     "grid.color": "0.9", "font.size": 9, "axes.edgecolor": "0.3"})
GREYS = ["0.0", "0.3", "0.5", "0.65"]; LS = ["-", "--", "-.", ":"]; MK = ["o", "s", "^", "D"]
Path("../results/figures").mkdir(parents=True, exist_ok=True)
Path("../results/tables").mkdir(parents=True, exist_ok=True)
def _save(fig, name):
    fig.savefig(f"../results/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"../results/figures/{name}.pdf", bbox_inches="tight")

df = pd.read_parquet("../data/synthetic_population.parquet")
ccols = [c for c in df.columns if c.startswith("c")]
C = df[ccols].to_numpy(); f = df["f"].to_numpy(); p = df["p"].to_numpy(); y = df["y"].to_numpy()
P = len(ccols); b0 = float((f - C.sum(1)).mean())
def quant(A, D): return D * np.round(A / D)
def fq(D): return b0 + quant(C, D).sum(1)

Deltas = np.array([0.02, 0.05, 0.1, 0.2, 0.4])
Ks = [5, 7, 10, 20]

def migration(fc, fqv, K):
    q = np.quantile(fc, np.linspace(0, 1, K + 1)[1:-1])
    return float(np.mean(np.digitize(fc, q) != np.digitize(fqv, q)))

rows = []
for D in Deltas:
    fqv = fq(D)
    for K in Ks:
        rows.append(dict(Delta=D, K=K, migration=migration(f, fqv, K)))
tab = pd.DataFrame(rows)
tab.to_csv("../results/tables/03_band_migration.csv", index=False)
print(tab.pivot(index="Delta", columns="K", values="migration").round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
for K, ls, col, mk in zip(Ks, LS, GREYS, MK):
    sub = tab[tab.K == K]
    ax[0].plot(sub.Delta, sub.migration, ls=ls, color=col, marker=mk, label=f"K={K}")
ax[0].set_xlabel(r"lattice spacing $\Delta$"); ax[0].set_ylabel("band migration rate")
ax[0].legend(fontsize=7)
for D, ls, col, mk in zip([0.05, 0.1, 0.2], LS, GREYS, MK):
    sub = tab[np.isclose(tab.Delta, D)]
    ax[1].plot(sub.K, sub.migration, ls=ls, color=col, marker=mk, label=rf"$\Delta$={D}")
ax[1].set_xlabel("number of bands K"); ax[1].set_ylabel("band migration rate")
ax[1].legend(fontsize=7)
fig.tight_layout(); _save(fig, "03_band_migration"); plt.close(fig)

K         5      7      10     20
Delta                            
0.02   0.013  0.016  0.027  0.055
0.05   0.036  0.051  0.076  0.151
0.10   0.068  0.095  0.140  0.287
0.20   0.144  0.216  0.297  0.516
0.40   0.403  0.509  0.678  0.851
